# Stage 3 — DPO Preference Alignment (Healthcare FAQ Assistant)

**Goal:** use Direct Preference Optimization to make the SFT model *prefer* answers that are correct, helpful, safe, professional and domain-specific over weak/unsafe/generic ones (`data/preference_dataset.jsonl`, 50+ triples).

Pipeline: Base → Stage 1 → Stage 2: SFT → **[Stage 3: DPO]** → Final Assistant

> ⚠️ Educational project — general health information only, not medical advice.

## 0. Install dependencies (Colab)

In [ ]:
# Run once on a fresh Colab GPU runtime (Runtime -> Change runtime type -> T4 GPU).
# Unsloth installs compatible transformers / peft / trl / bitsandbytes.
%%capture
!pip install -q unsloth
!pip install -q --no-deps "trl<0.12" peft accelerate bitsandbytes

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))

## 0b. Colab bootstrap — get repo files & set REPO_DIR

In [ ]:
# ============================================================
#  COLAB BOOTSTRAP  --  make repo files available + set REPO_DIR
#  Pick ONE method by setting BOOTSTRAP below.
# ============================================================
import os

BOOTSTRAP = "drive"   # "drive" (recommended) | "clone" | "local"

if BOOTSTRAP == "drive":
    # 1) Copy the `healthcare-ai-assistant-finetuning` folder into your Google Drive.
    # 2) Adjust the path below if you placed it somewhere other than MyDrive root.
    #    Drive is recommended because outputs/ persist across sessions, so Stage 1->2->3 chain.
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["REPO_DIR"] = "/content/drive/MyDrive/healthcare-ai-assistant-finetuning"

elif BOOTSTRAP == "clone":
    # Ephemeral: /content is wiped on disconnect. Run all stages in one session,
    # or set PUSH_TO_HUB=True so each stage's model is saved to the Hugging Face Hub.
    REPO_URL = "https://github.com/your-username/healthcare-faq-assistant.git"
    DEST = "/content/healthcare-faq-assistant"
    if not os.path.isdir(DEST):
        os.system(f"git clone {REPO_URL} {DEST}")
    os.environ["REPO_DIR"] = DEST

else:  # "local" -- running outside Colab, from inside the notebooks/ folder
    os.environ["REPO_DIR"] = ".."

print("REPO_DIR =", os.environ.get("REPO_DIR"))
assert os.path.isdir(os.path.join(os.environ["REPO_DIR"], "data")), \
    "REPO_DIR is wrong: no data/ folder found. Fix the path in this cell."

## 1. Select base model (must match the model used for SFT)

In [ ]:
# ============================================================
#  MODEL SELECTION  --  change MODEL_NAME to switch base model
# ============================================================
MODEL_OPTIONS = {
    "qwen2.5-0.5b":   "unsloth/Qwen2.5-0.5B",
    "llama-3.2-1b":   "unsloth/Llama-3.2-1B",
    "qwen2.5-1.5b":   "unsloth/Qwen2.5-1.5B",
    "tinyllama-1.1b": "unsloth/tinyllama",
    "gemma-2-2b":     "unsloth/gemma-2-2b",
}

MODEL_NAME = "qwen2.5-0.5b"   # <-- change this one line to pick a model
MODEL_REPO = MODEL_OPTIONS[MODEL_NAME]
print(f"Selected model: {MODEL_NAME}  ->  {MODEL_REPO}")

## 2. Paths & system prompt

In [ ]:
import os

# If you cloned the repo in Colab, point REPO_DIR at the repo root.
# This notebook lives in <repo>/notebooks/, so the repo root is one level up.
REPO_DIR   = os.environ.get("REPO_DIR", "..")
DATA_DIR   = os.path.join(REPO_DIR, "data")
OUTPUT_DIR = os.path.join(REPO_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Data dir:  ", os.path.abspath(DATA_DIR))
print("Output dir:", os.path.abspath(OUTPUT_DIR))

In [ ]:
# Shared system prompt used for instruction formatting / inference
SYSTEM_PROMPT = (
    "You are a Healthcare FAQ Assistant. You provide clear, general health information for "
    "educational purposes only. You are not a substitute for professional medical advice, "
    "diagnosis, or treatment. Always recommend consulting a qualified healthcare professional, "
    "and advise seeking emergency care for urgent symptoms."
)

## 2b. Hugging Face Hub config (optional)

In [ ]:
# ============================================================
#  HUGGING FACE HUB  --  set these to push your trained models
# ============================================================
PUSH_TO_HUB  = False                 # set True to upload after training
HF_USERNAME  = "your-hf-username"     # <-- your Hugging Face username
HF_TOKEN     = ""                     # <-- a WRITE token from https://huggingface.co/settings/tokens

# In Colab you can store the token as a secret instead of pasting it:
#   from google.colab import userdata
#   HF_TOKEN = userdata.get('HF_TOKEN')

if PUSH_TO_HUB and HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)
    print("Logged in to Hugging Face Hub as", HF_USERNAME)
else:
    print("PUSH_TO_HUB disabled (or no token). Models will be saved locally only.")

## 3. Load the SFT model

We load `outputs/stage2_merged` (the Stage-2 SFT model). If it is missing we fall back to the base model so the notebook still runs end-to-end.

In [ ]:
from unsloth import FastLanguageModel, PatchDPOTrainer
PatchDPOTrainer()   # must be called before building the DPOTrainer
import torch, os

max_seq_length = 2048
STAGE2_MERGED = os.path.join(OUTPUT_DIR, "stage2_merged")
load_from = STAGE2_MERGED if os.path.isdir(STAGE2_MERGED) else MODEL_REPO
print("Loading SFT model from:", load_from)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = load_from,
    max_seq_length = max_seq_length,
    dtype          = None,
    load_in_4bit   = True,
)

## 4. Set the chat template

In [ ]:
from unsloth.chat_templates import get_chat_template
if tokenizer.chat_template is None:
    tokenizer = get_chat_template(tokenizer, chat_template="chatml")
    print("Applied fallback ChatML template.")
else:
    print("Using the models built-in chat template.")

## 5. Attach LoRA adapters for DPO

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 6. Load & format the preference dataset

DPO needs three fields per row: **prompt**, **chosen**, **rejected**. The prompt is rendered with the chat template (ending with the assistant generation prompt); chosen/rejected are the answer texts.

In [ ]:
import json, os
from datasets import Dataset

path = os.path.join(DATA_DIR, "preference_dataset.jsonl")
rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
print(f"Loaded {len(rows)} preference examples")

EOS = tokenizer.eos_token
def to_dpo(ex):
    prompt = tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": ex["prompt"]},
        ],
        tokenize=False, add_generation_prompt=True,
    )
    return {
        "prompt":   prompt,
        "chosen":   ex["chosen"]   + EOS,
        "rejected": ex["rejected"] + EOS,
    }

dpo_dataset = Dataset.from_list([to_dpo(r) for r in rows])
print("\nExample prompt:\n", dpo_dataset[0]["prompt"][:400])
print("\nChosen:  ", dpo_dataset[0]["chosen"][:120])
print("Rejected:", dpo_dataset[0]["rejected"][:120])

## 7. Configure and run DPO training

In [ ]:
from trl import DPOTrainer, DPOConfig
from unsloth import is_bfloat16_supported

dpo_trainer = DPOTrainer(
    model = model,
    ref_model = None,           # Unsloth/PEFT uses the frozen base as the implicit reference
    train_dataset = dpo_dataset,
    tokenizer = tokenizer,      # newer TRL: rename to processing_class=tokenizer
    args = DPOConfig(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.1,
        num_train_epochs = 1,
        learning_rate = 5e-6,         # DPO uses a much smaller LR than SFT
        beta = 0.1,                   # DPO temperature
        max_length = 1024,
        max_prompt_length = 512,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.0,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = os.path.join(OUTPUT_DIR, "stage3_logs"),
        report_to = "none",
    ),
)
dpo_stats = dpo_trainer.train()
dpo_stats

## 8. Save the DPO-aligned (final) model

In [ ]:
STAGE3_ADAPTER = os.path.join(OUTPUT_DIR, "stage3_dpo")
STAGE3_MERGED  = os.path.join(OUTPUT_DIR, "stage3_merged")

model.save_pretrained(STAGE3_ADAPTER)
tokenizer.save_pretrained(STAGE3_ADAPTER)
print("Saved DPO adapter ->", STAGE3_ADAPTER)

model.save_pretrained_merged(STAGE3_MERGED, tokenizer, save_method="merged_16bit")
print("Saved final merged model ->", STAGE3_MERGED)

### Optional — push the Stage model to the Hugging Face Hub

Runs only if `PUSH_TO_HUB = True` and a write token is set above. Pushes both the LoRA adapter (small) and the merged 16-bit model.

In [ ]:
if PUSH_TO_HUB and HF_TOKEN:
    repo_adapter = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage3-dpo"
    repo_merged  = f"{HF_USERNAME}/healthcare-faq-{MODEL_NAME}-stage3-dpo-merged"

    # LoRA adapter
    model.push_to_hub(repo_adapter, token=HF_TOKEN)
    tokenizer.push_to_hub(repo_adapter, token=HF_TOKEN)
    print("Pushed adapter ->", repo_adapter)

    # Merged 16-bit model (ready for inference / vLLM / TGI)
    model.push_to_hub_merged(repo_merged, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
    print("Pushed merged model ->", repo_merged)
else:
    print("Skipped Hub upload (PUSH_TO_HUB is False or no token).")

### Reload this model later from the Hugging Face Hub

Once pushed, you (or anyone) can load it directly — no retraining needed:

```python
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage3-dpo-merged",
    max_seq_length = 2048, load_in_4bit = True,
)
```

Or with plain Transformers (CPU or GPU):

```python
from transformers import AutoModelForCausalLM, AutoTokenizer
repo = "your-hf-username/healthcare-faq-qwen2.5-0.5b-stage3-dpo-merged"
tokenizer = AutoTokenizer.from_pretrained(repo)
model = AutoModelForCausalLM.from_pretrained(repo)
```

## 9. Test the model after DPO

In [ ]:
FastLanguageModel.for_inference(model)

def ask(question, max_new_tokens=200):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors='pt'
    ).to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=max_new_tokens, use_cache=True,
                         do_sample=False, temperature=0.0)
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

for q in [
    "Can I stop my blood pressure medication if I feel fine?",
    "My 2-month-old baby has a fever, what should I do?",
    "Should I take antibiotics for a common cold?",
]:
    print("Q:", q)
    print("A:", ask(q))
    print("-" * 80)

## Done — Stage 3 complete ✅ (Final Healthcare FAQ Assistant)

The final model is in `outputs/stage3_merged`. Use it from `src/inference.py`, and fill in the comparison reports (`reports/`) using the base, SFT and DPO answers.